In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path
from torchvision import transforms as T

import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torchvision.models.detection.transform import GeneralizedRCNNTransform
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

In [10]:
class PolesDataset(Dataset):
    def __init__(self, txt_file, dataset_root, transforms=None):
        self.root = Path(dataset_root)
        self.transforms = transforms
        with open(self.root / txt_file) as f:
            self.img_paths = [l.strip() for l in f if l.strip()]

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        # Load image
        img_path = self.root / self.img_paths[idx]
        img = Image.open(img_path).convert("RGB")
        W, H = img.size

        # Load label — YOLO format: class cx cy w h (normalised)
        lbl_path = img_path.with_suffix(".txt").as_posix()
        lbl_path = Path(lbl_path.replace("images", "labels"))

        boxes  = []
        labels = []
        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        _, cx, cy, bw, bh = map(float, parts)
                        # Convert to absolute x1 y1 x2 y2
                        x1 = (cx - bw / 2) * W
                        y1 = (cy - bh / 2) * H
                        x2 = (cx + bw / 2) * W
                        y2 = (cy + bh / 2) * H
                        boxes.append([x1, y1, x2, y2])
                        labels.append(1)  # 1 = pole (0 is background in RCNN)

        # Handle empty images
        if len(boxes) == 0:
            boxes  = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,),   dtype=torch.int64)
        else:
            boxes  = torch.tensor(boxes,  dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels}

        if self.transforms:
            img = self.transforms(img)

        return img, target


In [13]:
train_transform = T.Compose([
T.ToTensor(),
T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
T.RandomHorizontalFlip(p=0.5),
])
val_transform = T.Compose([T.ToTensor()])

DATASET_ROOT = "Poles2025/Road_poles_iPhone"

train_dataset = PolesDataset("Train.txt",      DATASET_ROOT, transforms=train_transform)
val_dataset   = PolesDataset("Validation.txt", DATASET_ROOT, transforms=val_transform)

# collate_fn needed because images have different numbers of boxes
def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False, collate_fn=collate_fn)

print(f"Train: {len(train_dataset)} images")
print(f"Val:   {len(val_dataset)} images")

Train: 942 images
Val:   261 images


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

# Load pretrained, replace head for 1 class (pole) + background
model = fasterrcnn_mobilenet_v3_large_fpn(pretrained=True)
model.transform = GeneralizedRCNNTransform(
    min_size=1024,
    max_size=1024,
    image_mean=[0.485, 0.456, 0.406],
    image_std=[0.229, 0.224, 0.225]
)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes=2)  # 2 = background + pole
model.to(device)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=0.0001, weight_decay=0.0005
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=100, eta_min=1e-6
)
num_epochs = 100  # start with 20, RCNN is slower than YOLO
best_map = 0.0
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for imgs, targets in train_loader:
        imgs    = [img.to(device) for img in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(imgs, targets)
        losses    = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()

    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}  loss: {avg_loss:.4f}")
    if (epoch + 1) % 5 == 0:
        model.eval()
        metric = MeanAveragePrecision()
        with torch.no_grad():
            for imgs, targets in val_loader:
                imgs    = [img.to(device) for img in imgs]
                outputs = model(imgs)
                preds = [{"boxes": o["boxes"].cpu(), "scores": o["scores"].cpu(), "labels": o["labels"].cpu()} for o in outputs]
                tgts  = [{"boxes": t["boxes"].cpu(), "labels": t["labels"].cpu()} for t in targets]
                metric.update(preds, tgts)
        
        results = metric.compute()
        print(f"  mAP@50: {results['map_50']:.4f}  mAP@0.5:0.95: {results['map']:.4f}")
        
        # Save best model
        if results['map'].item() > best_map:
            best_map = results['map'].item()
            torch.save(model.state_dict(), "faster_rcnn_best.pth")
            print(f"best model saved")


# Save the model
torch.save(model.state_dict(), "faster_rcnn_poles_best_iphone.pth")
print("Model saved.")

Using: cuda


KeyboardInterrupt: 

In [ ]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Rebuild model architecture
model = fasterrcnn_resnet50_fpn(pretrained=False)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes=2)

# Load saved weights
model.load_state_dict(torch.load("faster_rcnn_poles_best_iphone.pth", map_location=device))
model.to(device)
model.eval()

print("Model loaded successfully")

/home/gustavga/.local/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/gustavga/.local/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /home/gustavga/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100.0%


Model loaded successfully


In [8]:
import torch
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from pathlib import Path

# Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()
model.to(device)

metric = MeanAveragePrecision(class_metrics=True)

with torch.no_grad():
    for imgs, targets in val_loader:
        imgs    = [img.to(device) for img in imgs]
        outputs = model(imgs)

        # Format predictions
        preds = [{
            "boxes":  o["boxes"].cpu(),
            "scores": o["scores"].cpu(),
            "labels": o["labels"].cpu()
        } for o in outputs]

        # Format targets
        tgts = [{
            "boxes":  t["boxes"].cpu(),
            "labels": t["labels"].cpu()
        } for t in targets]

        metric.update(preds, tgts)

results = metric.compute()

print("\n===== FASTER R-CNN RESULTS =====")
print(f"mAP@50:       {results['map_50']:.4f}")
print(f"mAP@0.5:0.95: {results['map']:.4f}")
print(f"Precision:    {results['map_per_class']:.4f}")
print(f"Recall:       {results['mar_100']:.4f}")


===== FASTER R-CNN RESULTS =====
mAP@50:       0.9859
mAP@0.5:0.95: 0.7796
Precision:    0.7796
Recall:       0.8097


In [9]:
from pathlib import Path
from PIL import Image
import torch
from torchvision import transforms as T

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

transform = T.Compose([T.ToTensor()])

test_img_dir = Path("Poles2025/Road_poles_iPhone/data/images/Test/test")
output_dir   = Path("runs/faster_rcnn/test_predictions/labels")
output_dir.mkdir(parents=True, exist_ok=True)

img_paths = sorted(test_img_dir.glob("*.jpg"))
print(f"Running inference on {len(img_paths)} test images...")

with torch.no_grad():
    for img_path in img_paths:
        img = Image.open(img_path).convert("RGB")
        W, H = img.size
        tensor = transform(img).unsqueeze(0).to(device)

        outputs = model(tensor)[0]

        boxes   = outputs["boxes"].cpu()
        scores  = outputs["scores"].cpu()
        labels  = outputs["labels"].cpu()

        # Save in YOLO format: class cx cy w h confidence
        txt_path = output_dir / img_path.with_suffix(".txt").name
        with open(txt_path, "w") as f:
            for box, score, label in zip(boxes, scores, labels):
                if score < 0.25:  # confidence threshold
                    continue
                x1, y1, x2, y2 = box
                cx = ((x1 + x2) / 2) / W
                cy = ((y1 + y2) / 2) / H
                bw = (x2 - x1) / W
                bh = (y2 - y1) / H
                # class is 0 (pole) — RCNN uses 1 internally but leaderboard expects 0
                f.write(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f} {score:.6f}\n")

print(f"Done! Predictions saved to {output_dir}")

Running inference on 138 test images...
Done! Predictions saved to runs/faster_rcnn/test_predictions/labels
